# 03 — Data Cleaning
Clean all 5 datasets and save processed versions to data/processed/.

In [1]:

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

RAW = r'../data/raw'
PROC = r'../data/processed'
os.makedirs(PROC, exist_ok=True)

ea = pd.read_csv(f'{RAW}/employee_performance_pro.csv')
ee = pd.read_csv(f'{RAW}/Employee_Performance_Dataset.csv')
occ = pd.read_csv(f'{RAW}/occupation_data.csv')
ess = pd.read_csv(f'{RAW}/essential_skills.csv')
sw = pd.read_csv(f'{RAW}/software_skills.csv')
print(f"Loaded raw files. ea shape: {ea.shape}, ee shape: {ee.shape}")


Loaded raw files. ea shape: (500, 24), ee shape: (5000, 13)


In [2]:

# ── Clean employee_performance_pro.csv ──
print("=== Cleaning employee_performance_pro ===")
print(f"Before: {ea.shape}")

# Drop exact duplicate rows
ea = ea.drop_duplicates()
print(f"After dedup: {ea.shape}")

# Fix JoiningDate to datetime
ea['JoiningDate'] = pd.to_datetime(ea['JoiningDate'], errors='coerce')

# Fix LastLeaveDate to datetime
ea['LastLeaveDate'] = pd.to_datetime(ea['LastLeaveDate'], errors='coerce')

# Normalize AttritionRisk to binary int
ea['AttritionRisk_Label'] = ea['AttritionRisk'].str.strip().str.lower().map({'yes': 1, 'no': 0})
missing_target = ea['AttritionRisk_Label'].isna().sum()
print(f"Missing AttritionRisk after mapping: {missing_target}")

# Normalize categoricals: strip whitespace, title case
for col in ['Gender', 'Department', 'JobRole']:
    ea[col] = ea[col].astype(str).str.strip().str.title()
# EducationLevel is numeric — leave as-is

# Fix numeric columns: clip negative values
ea['OvertimeHoursPerMonth'] = ea['OvertimeHoursPerMonth'].clip(lower=0)
ea['LeavesTaken'] = ea['LeavesTaken'].clip(lower=0)
ea['TrainingHours'] = ea['TrainingHours'].clip(lower=0)
ea['ProjectsHandled'] = ea['ProjectsHandled'].clip(lower=0)

# Handle missing numerics with median imputation
num_cols = ['MonthlySalary','OvertimeHoursPerMonth','LeavesTaken','ProjectsHandled',
            'TrainingHours','CustomerSatisfaction','WorkLifeBalanceScore','PerformanceRating',
            'YearsAtCompany','LastPromotionYear']
for c in num_cols:
    if ea[c].isna().any():
        med = ea[c].median()
        ea[c] = ea[c].fillna(med)
        print(f"  Imputed {c} with median={med:.2f}")

print(f"After cleaning: {ea.shape}")
print(f"Missing values remaining:\n{ea.isnull().sum()[ea.isnull().sum()>0]}")


=== Cleaning employee_performance_pro ===
Before: (500, 24)
After dedup: (500, 24)
Missing AttritionRisk after mapping: 0
  Imputed CustomerSatisfaction with median=5.00
After cleaning: (500, 25)
Missing values remaining:
Series([], dtype: int64)


In [3]:

# ── Clean Employee_Performance_Dataset.csv ──
print("=== Cleaning Employee_Performance_Dataset ===")
ee = ee.drop_duplicates()

# Normalize column names
ee.columns = ee.columns.str.strip()

# Normalize categoricals
str_cols = ['Department', 'Job Role', 'Promotion Eligibility']
for col in str_cols:
    if col in ee.columns and ee[col].dtype == object:
        ee[col] = ee[col].astype(str).str.strip().str.title()

# Clip attendance to [0, 100]
ee['Attendance (%)'] = ee['Attendance (%)'].clip(0, 100)

# Impute missing numerics
for c in ee.select_dtypes(include='number').columns:
    if ee[c].isna().any():
        ee[c] = ee[c].fillna(ee[c].median())
        print(f"  Imputed {c}")

print(f"Cleaned engagement shape: {ee.shape}")


=== Cleaning Employee_Performance_Dataset ===
Cleaned engagement shape: (5000, 13)


In [4]:

# ── Clean occupation_data.csv ──
print("=== Cleaning occupation_data ===")
occ = occ.drop_duplicates(subset=['O*NET-SOC Code'])
occ['Title'] = occ['Title'].str.strip()
occ['Description'] = occ['Description'].str.strip()
print(f"Occupation master shape: {occ.shape}")


=== Cleaning occupation_data ===
Occupation master shape: (1016, 3)


In [5]:

# ── Clean essential_skills.csv — filter to IM only ──
print("=== Cleaning essential_skills (IM only) ===")
print(f"Before filter: {ess.shape}")
ess_im = ess[ess['Scale ID'] == 'IM'].copy()
ess_im = ess_im.drop_duplicates()
ess_im['Element Name'] = ess_im['Element Name'].str.strip()
ess_im['Title'] = ess_im['Title'].str.strip()
# Drop rows where Data Value is missing
ess_im = ess_im.dropna(subset=['Data Value'])
print(f"After IM filter + clean: {ess_im.shape}")
print(f"Data Value range: [{ess_im['Data Value'].min():.2f}, {ess_im['Data Value'].max():.2f}]")


=== Cleaning essential_skills (IM only) ===
Before filter: (18200, 15)
After IM filter + clean: (9100, 15)
Data Value range: [1.00, 5.00]


In [6]:

# ── Clean software_skills.csv ──
print("=== Cleaning software_skills ===")
sw = sw.drop_duplicates()
sw['Element Name'] = sw['Element Name'].str.strip()
sw['Workplace Example'] = sw['Workplace Example'].str.strip()
sw['Title'] = sw['Title'].str.strip()
# Drop rows with missing Workplace Example (actual tool name)
sw = sw.dropna(subset=['Workplace Example'])
print(f"Software skills shape: {sw.shape}")


=== Cleaning software_skills ===
Software skills shape: (31821, 7)


In [7]:

# ── Save all processed files ──
ea.to_csv(f'{PROC}/employee_attrition_processed.csv', index=False)
ee.to_csv(f'{PROC}/engagement_processed.csv', index=False)
occ.to_csv(f'{PROC}/occupation_master.csv', index=False)
ess_im.to_csv(f'{PROC}/essential_skills_processed.csv', index=False)
sw.to_csv(f'{PROC}/software_skills_processed.csv', index=False)

print("\n=== SAVED PROCESSED FILES ===")
for fname in ['employee_attrition_processed.csv','engagement_processed.csv',
              'occupation_master.csv','essential_skills_processed.csv','software_skills_processed.csv']:
    path = f'{PROC}/{fname}'
    df = pd.read_csv(path)
    print(f"  {fname}: {df.shape}")



=== SAVED PROCESSED FILES ===
  employee_attrition_processed.csv: (500, 25)
  engagement_processed.csv: (5000, 13)
  occupation_master.csv: (1016, 3)
  essential_skills_processed.csv: (9100, 15)
  software_skills_processed.csv: (31821, 7)


**Cleaning complete.** 5 processed files saved. IM-only filter applied to essential_skills (removes LV rows to avoid averaging across incompatible scales).